# V1 — Dissertation Baseline (End-to-End) 🇬🇧📈

This notebook mirrors the V1 build pipeline:

1. Generate a reproducible intraday 1-minute session snapshot (offline)
2. Compute session KPIs
3. Fit **ARIMA(5,1,0)** and forecast the next 10 minutes
4. Fit a compact **LSTM** and forecast the next 10 minutes
5. Compare models + run Data Quality checks
6. Export the **7 dashboard pages** to `docs/dashboards/V1/exports/`

> Not financial advice.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('..').resolve().parents[0]
# Ensure src/ is on path
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd

from ftse100.config import (
    DOCS_DASH_V1_EXPORTS_DIR,
    V1_DATA_PROCESSED_DIR,
    V1_DATA_RAW_DIR,
    V1_FORECASTS_DIR,
    V1_METRICS_DIR,
)
from ftse100.data.io import save_intraday_snapshot, read_intraday_clean
from ftse100.data.synthetic import DailyOHLCV
from ftse100.features import compute_session_kpis
from ftse100.models.arima_model import fit_arima_forecast
from ftse100.models.lstm_model import fit_lstm_forecast
from ftse100.models.compare import compare_metrics
from ftse100.monitoring.dq import run_dq_checks, dq_summary_table, overall_status
from ftse100.viz.export_v1 import (
    export_page_01_market_overview,
    export_page_02_candles_volume,
    export_page_03_moving_averages,
    export_page_04_arima_forecast,
    export_page_05_lstm_forecast,
    export_page_06_model_comparison,
    export_page_07_dq_snapshot,
)
from ftse100.utils import make_run_id

run_id = make_run_id('v1_nb')
run_id

## 1) Snapshot dataset (offline)

The repo ships with an offline reproducible dataset. In a live environment you could replace this step with a provider pull.


In [ ]:
daily = DailyOHLCV(
    date='2026-02-13',
    open=10402.48,
    high=10454.54,
    low=10380.87,
    close=10446.35,
    volume=660_022_612,
)

raw_csv = V1_DATA_RAW_DIR / 'ftse100_intraday_1m_raw.csv'
clean_parquet = V1_DATA_PROCESSED_DIR / 'ftse100_intraday_1m_clean.parquet'
meta_json = V1_DATA_PROCESSED_DIR / 'ftse100_intraday_1m_metadata.json'

df = save_intraday_snapshot(daily, raw_csv, clean_parquet, meta_json, seed=42)
df.head()

## 2) KPIs

In [ ]:
dclean = read_intraday_clean(clean_parquet)

kpis = compute_session_kpis(dclean)
kpis

## 3) ARIMA forecast (10 minutes)

In [ ]:
arima_res = fit_arima_forecast(dclean, order=(5,1,0), horizon=10)
arima_res.metrics, arima_res.forecast.head()

## 4) LSTM forecast (10 minutes)

In [ ]:
lstm_res = fit_lstm_forecast(dclean, lookback=60, horizon=10, epochs=25, hidden_size=32, lr=3e-3, seed=42, return_model=False)
lstm_res.metrics, lstm_res.forecast.head()

## 5) Model comparison

In [ ]:
comp = compare_metrics(arima_res.metrics, lstm_res.metrics)
comp

## 6) Data Quality checks

In [ ]:
checks = run_dq_checks(dclean)
status = overall_status(checks)
dq = dq_summary_table(checks)
status, dq

## 7) Export the 7 dashboard pages

This reproduces the static exports in `docs/dashboards/V1/exports/`.


In [ ]:
DOCS_DASH_V1_EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

export_page_01_market_overview(dclean, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_01_market_overview.png', run_id)
export_page_02_candles_volume(dclean, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_02_candles_volume.png', run_id)
export_page_03_moving_averages(dclean, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_03_moving_averages.png', run_id)
export_page_04_arima_forecast(dclean, arima_res.forecast, arima_res.metrics, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_04_arima_forecast.png', run_id, order=arima_res.order)

# LSTM interpretability is produced in the scripted build; in the notebook we export without IG to keep runtime light.
export_page_05_lstm_forecast(dclean, lstm_res.forecast, lstm_res.metrics, lstm_res.training_history, None, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_05_lstm_forecast.png', run_id)

export_page_06_model_comparison(dclean, arima_res.forecast, lstm_res.forecast, comp, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_06_model_comparison.png', run_id)
export_page_07_dq_snapshot(dclean, DOCS_DASH_V1_EXPORTS_DIR / 'v1_page_07_data_quality_snapshot.png', run_id)

sorted([p.name for p in DOCS_DASH_V1_EXPORTS_DIR.glob('v1_page_*.png')])